# Collect agent traces on Kaggle (T4 + Ollama)

Launcher only: every step calls a CLI from the `atl` package.

**Before running (notebook sidebar → Settings):**
1. **Accelerator:** GPU T4 x2 (one GPU is used; the run waits on the Gemma API, not the GPU).
2. **Internet:** on.
3. **Persistence:** "Files only", so `/kaggle/working` (traces, cache, cost log) survives between
   sessions and a rerun resumes. Without it, download `traces.zip` (last cell) before the session ends.
4. **Add-ons → Secrets:** add `GEMINI_API_KEY` (the **free-tier** key) and attach it to this notebook.

In [ ]:
CONFIG = "configs/agent/collect_retail_qwen3_gemma.yaml"   # or collect_airline_qwen3_gemma.yaml
RUN_SECONDS = 3600                                           # stop after this long; rerun to continue
REPO = "https://github.com/Vivekananda-S/agent-trust-layer.git"
WORK = "/kaggle/working"

In [ ]:
import os, sys
print(sys.version)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Clone or update the repo inside the persistent working directory.
%cd {WORK}
!test -d agent-trust-layer && git -C agent-trust-layer pull --ff-only || git clone -q {REPO}
%cd {WORK}/agent-trust-layer
!mkdir -p {WORK}/data && (test -L data || (rm -rf data && ln -s {WORK}/data data)) && ls -la data

In [ ]:
# The package and tau2-bench need Python 3.12. If the image is older, build a 3.12 venv with uv
# and put it first on PATH, so `!atl-agent ...` uses it.
if sys.version_info < (3, 12):
    !pip install -q uv
    !uv venv -q --python 3.12 {WORK}/venv312
    !uv pip install -q --python {WORK}/venv312/bin/python -e ".[agent]"
    os.environ["PATH"] = f"{WORK}/venv312/bin:" + os.environ["PATH"]
else:
    !pip install -q -e ".[agent]"
!which atl-agent && atl-agent --help | head -3

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")

In [ ]:
# Ollama on GPU 0 only, 32k context, 3 parallel slots with an 8-bit KV cache (fits a T4).
!apt-get -qq install -y zstd pciutils > /dev/null
!command -v ollama || (curl -fsSL https://ollama.com/install.sh | sh)
!pkill -f "ollama serve" || true
import subprocess, time, urllib.request
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "OLLAMA_CONTEXT_LENGTH": "32768",
       "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_NUM_PARALLEL": "3", "OLLAMA_FLASH_ATTENTION": "1",
       "OLLAMA_KV_CACHE_TYPE": "q8_0"}
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("ollama up"); break
    except Exception:
        time.sleep(1)

In [ ]:
import yaml
MODEL_TAG = yaml.safe_load(open(CONFIG))["agent_model"].split("/", 1)[1]
!ollama pull {MODEL_TAG}
!atl-agent fetch-data
!atl-agent run --config {CONFIG} --smoke --smoke-dir /tmp/smoke 2>&1 | tail -1

In [ ]:
# The run: stops after RUN_SECONDS; finished runs are kept, rerun this cell to continue.
RUN_NAME = yaml.safe_load(open(CONFIG))["run_name"]
!mkdir -p data/logs
!timeout {RUN_SECONDS} atl-agent run --config {CONFIG} >> data/logs/{RUN_NAME}.log 2>&1
!atl-agent stats --run-dir data/traces/{RUN_NAME} 2>&1 | tail -1
!grep -c "429\|RESOURCE_EXHAUSTED" data/logs/{RUN_NAME}.log
!ollama ps

In [ ]:
# Download this (Output panel → traces.zip) to share results or to carry them to another session.
!cd data && zip -qr {WORK}/traces.zip traces logs costs.jsonl && ls -la {WORK}/traces.zip